In [0]:
from pyspark.sql import functions as F

sc = spark.table("workspace.enigh.silver_concentrado")
sg = spark.table("workspace.enigh.silver_gastos")

# 1) Gastos -> una fila por hogar
top = [r["grupo3"] for r in sg.groupBy("grupo3").agg(F.sum("gasto_total_tri").alias("t"))
                              .orderBy(F.desc("t")).limit(8).collect()]
print("Grupos de gasto usados:", top)

agg_exprs = [
    F.sum("gasto_total_tri").alias("gasto_total_tri"),
    F.count("*").alias("n_registros"),
    F.countDistinct("clave").alias("n_claves"),
    F.countDistinct("grupo3").alias("n_grupos"),
    F.sum(F.coalesce(F.col("gasto_nm_tri"), F.lit(0.0))).alias("_nm"),
] + [F.sum(F.when(F.col("grupo3") == g, F.col("gasto_total_tri")).otherwise(0.0)).alias(f"_g_{g}") for g in top]

gh = sg.groupBy("folioviv", "foliohog").agg(*agg_exprs)
gh = gh.withColumn("pct_no_monetario", F.col("_nm") / F.col("gasto_total_tri"))
for g in top:
    gh = gh.withColumn(f"pct_g_{g}", F.col(f"_g_{g}") / F.col("gasto_total_tri"))
gh = gh.drop("_nm", *[f"_g_{g}" for g in top])

# 2) Variable objetivo: hogar en el 20% de mayor ingreso
umbral = sc.approxQuantile("ing_cor", [0.8], 0.0001)[0]
base = (sc.select("folioviv", "foliohog", "entidad", "tam_loc", "est_socio", "clase_hog", "sexo_jefe",
                  "edad_jefe", "educa_jefe", "tot_integ", "ocupados", "ing_cor")
          .withColumn("entidad", F.col("entidad").cast("int"))
          .withColumn("ingreso_alto", (F.col("ing_cor") >= umbral).cast("int")))

# 3) Unir y guardar
gold = base.join(gh, ["folioviv", "foliohog"], "left")
cols_gasto = [c for c in gh.columns if c not in ("folioviv", "foliohog")]
gold = gold.fillna(0, subset=cols_gasto)

gold.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.enigh.gold_hogares")

# ---------- REPORTE ----------
g = spark.table("workspace.enigh.gold_hogares")
print("\ngold_hogares:", g.count(), "filas,", len(g.columns), "columnas")
print("Umbral de ingreso (percentil 80):", umbral)
print("Balance de clases:", {r["ingreso_alto"]: r["count"] for r in g.groupBy("ingreso_alto").count().collect()})
print("Hogares sin gastos registrados:", g.filter(F.col("n_registros") == 0).count())
print("Columnas:", g.columns)

Grupos de gasto usados: ['011', '046', '111', '072', '131', '083', '056', '045']

gold_hogares: 91414 filas, 26 columnas
Umbral de ingreso (percentil 80): 99708.17
Balance de clases: {0: 73131, 1: 18283}
Hogares sin gastos registrados: 0
Columnas: ['folioviv', 'foliohog', 'entidad', 'tam_loc', 'est_socio', 'clase_hog', 'sexo_jefe', 'edad_jefe', 'educa_jefe', 'tot_integ', 'ocupados', 'ing_cor', 'ingreso_alto', 'gasto_total_tri', 'n_registros', 'n_claves', 'n_grupos', 'pct_no_monetario', 'pct_g_011', 'pct_g_046', 'pct_g_111', 'pct_g_072', 'pct_g_131', 'pct_g_083', 'pct_g_056', 'pct_g_045']
